In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import psycopg2

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

print("Libraries imported.")

Libraries imported.


### connect to PostgreSQL

In [7]:
conn = psycopg2.connect(
    dbname="Marketing-HW2",
    user="postgres",
    password="my_secret",
    host="localhost",
    port=5433
)

print("Successfully connected to PostgreSQL.")

Successfully connected to PostgreSQL.


### load analytics views

In [8]:
rfm = pd.read_sql("SELECT * FROM analytics.rfm_base;", conn)
revenue = pd.read_sql("SELECT * FROM analytics.order_revenue;", conn)
customer_revenue = pd.read_sql("SELECT * FROM analytics.customer_revenue;", conn)
customer = pd.read_sql("SELECT * FROM core.customer;", conn)
orders = pd.read_sql("SELECT * FROM core.orders;", conn)

print("Data loaded from PostgreSQL.")

/tmp/ipykernel_15743/1808831651.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  rfm = pd.read_sql("SELECT * FROM analytics.rfm_base;", conn)
/tmp/ipykernel_15743/1808831651.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  revenue = pd.read_sql("SELECT * FROM analytics.order_revenue;", conn)
/tmp/ipykernel_15743/1808831651.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  customer_revenue = pd.read_sql("SELECT * FROM analytics.customer_revenue;", conn)
/tmp/ipykernel_15743/1808831651.py:4: UserWarning: pandas only s

Data loaded from PostgreSQL.


In [9]:
rfm["r_score"] = pd.qcut(
    rfm["recency"],
    5,
    labels=[5, 4, 3, 2, 1]
)

rfm["f_score"] = pd.qcut(
    rfm["frequency"].rank(method="first"),
    5,
    labels=[1, 2, 3, 4, 5]
)

rfm["m_score"] = pd.qcut(
    rfm["monetary"].rank(method="first"),
    5,
    labels=[1, 2, 3, 4, 5]
)

rfm["rfm_score"] = (
    rfm["r_score"].astype(int) * 100 +
    rfm["f_score"].astype(int) * 10 +
    rfm["m_score"].astype(int)
)

rfm.sort_values("rfm_score", ascending=False).head(10)

,customer_id,recency,frequency,monetary,r_score,f_score,m_score,rfm_score
1262,CUST001509,65.0,13,10751.38,5,5,5,555
8479,CUST009969,92.0,26,19759.28,5,5,5,555
5812,CUST006858,64.0,18,16933.71,5,5,5,555
1327,CUST001578,69.0,22,22611.71,5,5,5,555
1332,CUST001587,65.0,16,12066.32,5,5,5,555
5114,CUST006051,69.0,29,23303.72,5,5,5,555
5115,CUST006052,58.0,20,12590.42,5,5,5,555
7837,CUST009226,65.0,15,13072.70,5,5,5,555
7845,CUST009237,74.0,14,12128.66,5,5,5,555
1229,CUST001473,88.0,26,15841.78,5,5,5,555


In [10]:
analysis_date = pd.read_sql("""
    SELECT MAX(order_date) AS max_date
    FROM core.orders;
""", conn)["max_date"][0]

clv = customer_revenue.copy()

clv["last_purchase_date"] = pd.to_datetime(clv["last_purchase_date"])
analysis_date = pd.to_datetime(analysis_date)

clv["days_since_last_purchase"] = (
    analysis_date - clv["last_purchase_date"]
).dt.days

clv["is_churned"] = clv["days_since_last_purchase"] > 90

total_revenue = clv["total_revenue"].sum()
total_transactions = clv["number_of_transactions"].sum()
total_customers = clv["customer_id"].nunique()
churned_customers = clv["is_churned"].sum()

apv = total_revenue / total_transactions
pf = total_transactions / total_customers
churn_rate = churned_customers / total_customers

overall_clv = (apv * pf) / churn_rate

print("Average Purchase Value:", apv)
print("Purchase Frequency:", pf)
print("Churn Rate:", churn_rate)
print("Overall CLV:", overall_clv)

clv["avg_purchase_value"] = (
    clv["total_revenue"] / clv["number_of_transactions"]
)

clv["simple_clv"] = (
    clv["avg_purchase_value"] *
    clv["number_of_transactions"]
) / churn_rate

clv.sort_values("simple_clv", ascending=False).head(20)

Average Purchase Value: 858.7709357772394
Purchase Frequency: 6.693066980023501
Churn Rate: 0.8038777908343125
Overall CLV: 7150.105972810993


/tmp/ipykernel_15743/4259348871.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  analysis_date = pd.read_sql("""


,customer_id,number_of_transactions,first_purchase_date,last_purchase_date,total_revenue,avg_item_revenue,days_since_last_purchase,is_churned,avg_purchase_value,simple_clv
50,CUST000058,152,2023-05-17,2025-12-19,128361.85,755.069706,64,False,844.485855,159678.313624
2323,CUST002744,104,2023-08-28,2025-12-10,118002.07,1000.017542,73,False,1134.635288,146791.056234
1999,CUST002370,83,2023-04-20,2025-12-25,91195.67,980.598602,58,False,1098.743012,113444.694007
3023,CUST003565,66,2023-07-17,2025-12-21,73102.06,987.865676,62,False,1107.606970,90936.782722
6786,CUST008014,80,2023-05-13,2025-11-27,70803.35,804.583523,86,False,885.041875,88077.256030
3246,CUST003831,68,2023-08-11,2025-12-28,70660.05,942.134000,55,False,1039.118382,87898.995103
7265,CUST008575,81,2023-05-05,2025-12-15,61285.88,712.626512,68,False,756.615802,76237.807163
5710,CUST006740,59,2024-01-10,2025-12-14,59303.41,898.536515,69,False,1005.142542,73771.673600
6536,CUST007723,61,2023-04-27,2025-12-16,57659.36,835.642899,67,False,945.235410,71726.524426
5211,CUST006165,60,2023-03-04,2025-12-12,56845.18,823.843188,71,False,947.419667,70713.708785


In [11]:
model_data = rfm.merge(
    clv[["customer_id", "simple_clv"]],
    on="customer_id",
    how="inner"
)

X = model_data[["recency", "frequency", "monetary"]]
y = model_data["simple_clv"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

model = RandomForestRegressor(
    n_estimators=200,
    random_state=42
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("RMSE:", rmse)
print("MAE:", mae)
print("R2:", r2)

RMSE: 827.7162861427034
MAE: 25.729930638795672
R2: 0.9917880827579143


In [12]:
clv["clv_segment"] = pd.qcut(
    clv["simple_clv"],
    q=[0, 0.2, 0.5, 0.8, 1.0],
    labels=["Bronze", "Silver", "Gold", "Platinum"]
)

segment_summary = clv.groupby("clv_segment").agg(
    customers=("customer_id", "count"),
    avg_clv=("simple_clv", "mean"),
    avg_spending=("total_revenue", "mean"),
    avg_frequency=("number_of_transactions", "mean"),
    churn_rate=("is_churned", "mean")
).reset_index()

segment_summary

,clv_segment,customers,avg_clv,avg_spending,avg_frequency,churn_rate
0,Bronze,1702,660.461117,530.930024,1.740893,0.957109
1,Silver,2553,2816.649598,2264.242056,3.795535,0.880924
2,Gold,2553,7274.467095,5847.782538,6.815119,0.783000
3,Platinum,1702,19953.393707,16040.090053,15.808461,0.566392
